# Tutorial 0.1 — Getting Started: Your AI Workbench

*Chapter 0 — before Lab 1.1 · 20 minutes · individual*

Twenty minutes now saves you confusion in every lab that follows. You will touch the
exact machinery the whole course runs on: one helper module, one key you never see,
and one deterministic offline mode. Run every cell top to bottom.

## Objectives

By the end of this tutorial, you will:

- Verify your environment and tell **live** mode from **mock** mode.
- Make your first calls through `course_ai`: `chat`, `chat_json`, `embed` + `cosine`,
  and one tool-calling step.
- Read a lab's self-checks (`assert`) and reset any lab to a clean state.
- State the key-hygiene rules every lab in this course follows.


## How these labs work

- Every LLM call goes through **`course_ai`** — the one module between you and the
  OpenAI SDK (`chat`, `chat_json`, `chat_tools`, `embed`, `cosine`). It lives next to
  this notebook; read it, it is short.
- The API key comes from the environment (`OPENAI_API_KEY`, injected on the classroom
  VM). It is never pasted into a notebook and never printed.
- No key? Everything still runs: a deterministic **mock** answers, its outputs prefixed
  `[MOCK]`. Force it with `COURSE_AI_MOCK=1`.
- Labs check themselves: a failing cell raises `AssertionError` with a message that
  tells you what broke.

## Step 1 — Verify the environment (4 min)

One cell to prove the Python, the dependencies, and the mode you are running in. The
key is reported only as present-or-absent — never its value.

In [ ]:
import os
import sys

import course_ai

print("python:", sys.version.split()[0])
print("mode:", course_ai.mode())   # 'live (model: ...)' or 'mock (...)'
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

import openai  # noqa: F401
import pandas  # noqa: F401
import pytest  # noqa: F401
import sklearn  # noqa: F401
print("dependencies OK")

## Step 2 — Your first call (4 min)

One function, two worlds: with a key the model answers; without one the deterministic
mock stands in — so the same exercise works on every machine in the room, and every
notebook is safe to re-run.

In [ ]:
answer = course_ai.chat(
    "In one sentence: why does the context window matter to a developer?",
    system="You are a concise mentor for software engineers.")
print(answer)

## Step 3 — Structured output (4 min)

`chat` gives you prose; `chat_json` gives you data your code can assert on. You pass a
JSON Schema, the model fills it. Every eval, judge, and guardrail later in the course
builds on this one call.

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "category": {"type": "string"},
        "confidence": {"type": "string"},
    },
    "required": ["category", "confidence"],
    "additionalProperties": False,
}

result = course_ai.chat_json(
    "Classify this support ticket: 'Nightly sync stops at 87% every time.'",
    SCHEMA)
print(result)
assert "category" in result and "confidence" in result
print("schema honored — code can rely on the shape")

## Step 4 — Embeddings: meaning as numbers (4 min)

`embed` turns text into vectors; `cosine` measures how close two meanings are (1.0 =
identical direction). The live model encodes *meaning*; the offline mock approximates
it with word overlap — enough to watch the mechanics work. This pair of calls is what
search, retrieval, and RAG are built from.

In [ ]:
texts = [
    "password reset link never arrives",
    "users report the password reset link never arrives",
    "invoice totals include tax and shipping",
]
vecs = course_ai.embed(texts)

sim_same = course_ai.cosine(vecs[0], vecs[1])
sim_diff = course_ai.cosine(vecs[0], vecs[2])
print(f"same issue, different wording: {sim_same:.3f}")
print(f"unrelated topic:               {sim_diff:.3f}")
assert sim_same > sim_diff
print("\nsame-issue texts sit closer together — retrieval in one line")

## Step 5 — One tool call: a preview of Chapter 7 (4 min)

Agents are this same API with one addition: the model can ask **your code** to run a
tool. You describe each tool as a JSON schema; the model replies with a structured
request to call it. Here is a single step — in Lab 7.1 you will put it in a loop with
guardrails around it.

![The cognitive loop every agent runs — with its risk lens](diagrams/ch07_cognitive_loop.png)

*The cognitive loop every agent runs — with its risk lens (Chapter 7 deck).*

In [ ]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "search_code",
        "description": "Search the repository for a string.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
}]

msg = course_ai.chat_tools(
    [{"role": "user", "content": "Find where the billing total is calculated."}],
    TOOLS)

if getattr(msg, "tool_calls", None):
    call = msg.tool_calls[0]
    print("the model asked to run:", call.function.name)
    print("with arguments:", call.function.arguments)
    print("\nnote: it asked — YOUR code decides whether to execute. That gap is the guardrail.")
else:
    print("the model answered directly:", msg.content)

## Housekeeping — reset and key hygiene

- **Reset a lab:** delete its generated files (`tdd_workspace/`, `capstone_workspace/`,
  `agent_repo/`, `eval_scorecard.json`, `audit_report.json`), then
  *Kernel → Restart & Run All*.
- **Key hygiene:** the key lives in the environment — never in a notebook, never in a
  `print`. If you ever see one in a cell, treat it as compromised and tell the
  instructor.
- **Stuck?** Run `python verify_setup.py` from the pack root, then restart the kernel.

You are ready for **Lab 1.1 — Assistant Recon**. Have fun.